# **Attacks on text attribution, amnesic probing and an attack on the SAE representation**

Practice for the module [“Attacks on Explanations”](https://ai-interpretability.school), lessons “Attacks on text attribution” and “Probes and sparse autoencoder illusions”.

In the lesson “Attacks on text attribution” two attacks, TextExplanationFooler (TEF; Ivankay et al., ICLR 2022) and ExplainFooler (Sinha et al., BlackboxNLP 2021), changed the attribution of a text classifier by replacing a few words with synonyms while the predicted class stayed the same. In the lesson “Probes and sparse autoencoder illusions” the conclusion “a direction was found in the representation, so the model uses it” was checked by an intervention (amnesic probing, Elazar et al., 2021) and by an attack on the input (Li et al., 2026). Here all four methods are computed with our own code on open models:

- section 1: the DistilBERT sentiment classifier on SST-2 and Integrated Gradients (IG) attribution from `captum`;
- section 2: how much the attribution changes by itself, without an adversary: the IG baseline and the number of steps;
- section 3: ExplainFooler: word order by leave-one-out, the objective is the L2 distance between attributions, compared with a random choice of synonym;
- section 4: TEF: word order by the zero embedding, the objective is the Pearson correlation, the ACC area and the random attack RA;
- section 5: amnesic probing: INLP on a DistilBERT layer, removal of two properties and a control with random directions;
- section 6: an attack on the SAE representation of GPT-2 small by replacing one token.

**Why not TextAttack.** The lesson says that the TEF search is repeated on [TextAttack](https://github.com/QData/TextAttack), and ExplainFooler is written on it too. But TextAttack 0.3.11 pulls in `flair` at import, and `flair` requires `transformers` 4.x and installs in Colab with `transformers` 5 only by downgrading it. So the three components of the attack, synonym candidates, constraints and greedy search, are written here with our own code under the same rules: synonyms from WordNet, as in TextAttack's `WordSwapWordNet`, of the same part of speech, no stop words, each word replaced at most once, the predicted class preserved. WordNet does not have the counter-fitted embeddings from which both papers take synonyms, so there are fewer candidates.

**Environment.** The notebook was checked on Colab versions (`torch` 2.11, `transformers` 5.18); `captum`, `sae-lens` and `transformer-lens` are installed by the first cell. It runs on CPU, faster on GPU. Randomness enters only the choice of sentences and the random points of comparison; the seed is fixed.

In [ ]:
%pip install -q captum==0.9.0 nltk==3.9.1 transformer-lens==3.9.0 sae-lens==6.54.0

In [ ]:
import warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import wordnet as wn
from scipy.stats import spearmanr, pearsonr
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import IntegratedGradients, Saliency

for pkg in ('wordnet', 'stopwords', 'averaged_perceptron_tagger_eng'):
    nltk.download(pkg, quiet=True)
STOP = set(nltk.corpus.stopwords.words('english'))
warnings.filterwarnings('ignore', message='.*encountered in matmul')   # spurious numpy warnings on some processors

SEED = 0
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
tok = AutoTokenizer.from_pretrained(NAME)
clf = AutoModelForSequenceClassification.from_pretrained(NAME).to(device).eval()
emb_layer = clf.distilbert.embeddings.word_embeddings
sst = load_dataset('stanfordnlp/sst2', split='validation')
print('device:', device, '| sentences in the SST-2 validation split:', len(sst))

## 1. The classifier and word-level IG attribution

`distilbert-base-uncased-finetuned-sst-2-english` is DistilBERT fine-tuned on SST-2 sentiment; ExplainFooler computes its table 1 on this same model. We compute the attribution with IG from `captum` over token embeddings: the function `fwd` receives the embedding matrix and returns the logit of the required class, and DistilBERT adds the position embeddings itself. The default baseline is the embedding of the `[PAD]` token at all positions except `[CLS]` and `[SEP]`, 50 steps, as in ExplainFooler. TEF uses the zero embedding matrix as the baseline; this is `baseline='zero'`.

A word may split into several subword tokens. As in ExplainFooler, a word gets the score of its tokens with the largest absolute value; this way the attribution vectors before and after a word replacement have the same length, the number of words.

Words are the parts of a sentence separated by spaces: in SST-2 the text is already split into words and punctuation.

**Task 1. Compute the classifier's accuracy on all 872 sentences of the validation split. Then for the first sentence in `sents` check IG completeness at the token level: the sum of the attributions of all tokens, including `[CLS]` and `[SEP]`, must be close to the difference of the logits on the input and on the baseline.**

In [ ]:
@torch.no_grad()
def predict(sentences, bs=64):
    out = []
    for i in range(0, len(sentences), bs):
        enc = tok(sentences[i:i + bs], return_tensors='pt', padding=True).to(device)
        out.append(torch.softmax(clf(**enc).logits, -1).cpu())
    return torch.cat(out).numpy()


def fwd(emb, mask, target):
    return clf(inputs_embeds=emb, attention_mask=mask.expand(emb.shape[0], -1)).logits[:, target]


ig = IntegratedGradients(fwd)


def encode(words, baseline):
    enc = tok(words, is_split_into_words=True, return_tensors='pt').to(device)
    ids = enc['input_ids']
    with torch.no_grad():
        emb = emb_layer(ids)
        if baseline == 'pad':
            b_ids = torch.full_like(ids, tok.pad_token_id)
            b_ids[0, 0], b_ids[0, -1] = tok.cls_token_id, tok.sep_token_id
            base = emb_layer(b_ids)
        else:
            base = torch.zeros_like(emb)
    return enc, emb, base


def to_words(tok_attr, word_ids, n_words):
    # the score of a word is the score of its subword token with the largest absolute value
    out = np.zeros(n_words)
    for j, w in enumerate(word_ids):
        if w is not None and abs(tok_attr[j]) > abs(out[w]):
            out[w] = tok_attr[j]
    return out


def word_attr(words, target, n_steps=50, baseline='pad', zero_word=None):
    enc, emb, base = encode(words, baseline)
    wid = enc.word_ids()
    if zero_word is not None:          # TEF: the embedding of one word is replaced by zero
        emb = emb.clone()
        emb[0, [j for j, w in enumerate(wid) if w == zero_word]] = 0
    a = ig.attribute(emb, baselines=base, additional_forward_args=(enc['attention_mask'], target),
                     n_steps=n_steps, internal_batch_size=64)
    return to_words(a.sum(-1)[0].detach().cpu().numpy(), wid, len(words))


sents = [s for s in sst['sentence'] if 8 <= len(s.split()) <= 18]
rng = np.random.default_rng(SEED)
sample = [sents[i] for i in rng.choice(len(sents), 30, replace=False)]
w = sample[0].split()
label = int(predict([sample[0]])[0].argmax())
print('sentence:', sample[0], '| class:', clf.config.id2label[label])
print({x: round(float(v), 3) for x, v in zip(w, word_attr(w, label))})

In [ ]:
# Your code here

## 2. How much the attribution changes without an adversary

Before measuring how much an attack changes the attribution, it helps to know how much it changes because of the researcher's own choices. On the 30 sentences of `sample` we compare IG with the `[PAD]` baseline and with the zero baseline, and IG with 20 and 50 steps against IG with 200 steps. The measure is the Spearman rank correlation of word attribution vectors, as in table 1 of ExplainFooler.

In [ ]:
rho = {'[PAD] vs zero baseline': [], '20 vs 200 steps': [], '50 vs 200 steps': []}
for s in sample:
    w = s.split()
    lab = int(predict([s])[0].argmax())
    a200 = word_attr(w, lab, 200)
    rho['[PAD] vs zero baseline'].append(spearmanr(word_attr(w, lab), word_attr(w, lab, baseline='zero')).statistic)
    rho['20 vs 200 steps'].append(spearmanr(word_attr(w, lab, 20), a200).statistic)
    rho['50 vs 200 steps'].append(spearmanr(word_attr(w, lab, 50), a200).statistic)
for k, v in rho.items():
    print(f'{k:>24}: mean Spearman correlation {np.mean(v):.3f}, minimum {np.min(v):.3f}')

The number of steps barely changes the ranks: 20 steps are enough, and below we use 20 steps in the attack search so that it runs faster. The baseline changes the ranks strongly: the attributions with `[PAD]` and with the zero embedding have a mean Spearman correlation of 0.432, so they are different explanations of the same prediction. This is worth keeping in mind when reading the numbers below: an attack after which the correlation with the original attribution falls to the level of the row “[PAD] vs zero baseline” changes the explanation no more than the choice of baseline does.

## 3. ExplainFooler: synonym replacements against IG

Replacement candidates are WordNet synonyms of the same part of speech; the word must be in its base form (otherwise the synonym would be in a different form), and stop words and punctuation are not touched. The greedy search goes over the words in a given order; at each word it tries the synonyms, discards those that change the predicted class, and takes the best of the rest according to the attack objective. The function `attack` is shared by the two attacks; they differ in the word order and the objective.

In ExplainFooler the order is given by leave-one-out: a word is deleted and we look at how much the probability of the predicted class drops. The objective is the L2 norm of the difference of attribution vectors (L2 Norm, equation (3) of the paper); the point of comparison is a random synonym among those that passed the constraints. The share of replaced words grows up to 0.5.

There is no constraint on sentence meaning here, which the papers check with the Universal Sentence Encoder, and WordNet synonyms often come from a different sense of the word (“credits” → “quotation”). The classifier's prediction does not change by construction, but a human would notice such a replacement. Keep this in mind: an attack without a meaning constraint is stronger than the one in the papers.

In [ ]:
POS = {'NN': wn.NOUN, 'VB': wn.VERB, 'JJ': wn.ADJ, 'RB': wn.ADV}


def synonyms(words, k):
    cands = []
    for w, t in nltk.pos_tag(words):
        pos = POS[t[:2]] if t[:2] in POS else None
        if pos is None or w in STOP or not w.isalpha() or wn.morphy(w, pos) != w:
            cands.append([])
            continue
        syn = []
        for ss in wn.synsets(w, pos=pos):
            for lem in ss.lemma_names():
                lem = lem.lower()
                if lem.isalpha() and lem != w and lem not in syn:
                    syn.append(lem)
        cands.append(syn[:k])
    return cands


def attack(words, order, cands, objective, max_ratio, pick, baseline='pad', n_steps=20):
    """Greedy word replacement. Returns a list of steps: (share of replaced words, words, attribution)."""
    label = int(predict([' '.join(words)])[0].argmax())
    a0 = word_attr(words, label, n_steps, baseline)
    cur, steps = list(words), [(0.0, list(words), a0)]
    for pos in order:
        if len(steps) / len(words) > max_ratio:
            break
        opts = [cur[:pos] + [c] + cur[pos + 1:] for c in cands[pos]]
        if not opts:
            continue
        keep = predict([' '.join(o) for o in opts]).argmax(1) == label     # the class is preserved
        opts = [o for o, k in zip(opts, keep) if k]
        if not opts:
            continue
        if pick == 'random':
            cur = opts[rng.integers(len(opts))]
            a = word_attr(cur, label, n_steps, baseline)
        else:
            atts = [word_attr(o, label, n_steps, baseline) for o in opts]
            j = int(np.argmax([objective(a0, x) for x in atts]))
            cur, a = opts[j], atts[j]
        steps.append((len(steps) / len(words), list(cur), a))
    return steps


def l2(a0, a1):
    return np.linalg.norm(a0 - a1)          # L2 Norm of ExplainFooler, eq. (3): IG scores are not rescaled


def loo_order(words):
    label = int(predict([' '.join(words)])[0].argmax())
    p = predict([' '.join(words[:i] + words[i + 1:]) for i in range(len(words))])[:, label]
    return list(np.argsort(p))          # first the word whose removal lowers the confidence the most


print(sample[1])
print({w: c[:5] for w, c in zip(sample[1].split(), synonyms(sample[1].split(), 50)) if c})

In [ ]:
ef = {'L2': [], 'random': []}
for s in sample:
    w = s.split()
    cands, order = synonyms(w, 50), loo_order(w)
    for arm in ef:
        steps = attack(w, order, cands, l2, 0.5, 'best' if arm == 'L2' else 'random')
        ef[arm].append([(r, max(spearmanr(steps[0][2], a).statistic, 0), ' '.join(x)) for r, x, a in steps])

BINS = [(0, 0.1), (0.1, 0.2), (0.2, 0.3), (0.3, 0.4), (0.4, 0.5)]


def by_bin(runs):
    # for each sentence take the last step inside the interval of the share of replacements
    out = []
    for lo, hi in BINS:
        vals = [[c for r, c, _ in run if lo < r <= hi][-1] for run in runs if any(lo < r <= hi for r, _, _ in run)]
        out.append((np.mean(vals) if vals else np.nan, len(vals)))
    return out


for arm, runs in ef.items():
    print(arm, ' | '.join(f'{lo}-{hi}: {m:.3f} (n={n})' for (lo, hi), (m, n) in zip(BINS, by_bin(runs))))
best = min(range(len(sample)), key=lambda i: ef['L2'][i][-1][1])
print('\nbefore the attack:', ef['L2'][best][0][2])
print('after the attack: ', ef['L2'][best][-1][2], f'| Spearman {ef["L2"][best][-1][1]:.3f}')

In [ ]:
centers = [(lo + hi) / 2 for lo, hi in BINS]
plt.figure(figsize=(5.5, 3.6))
for arm, color in (('L2', '#c0392b'), ('random', '#7f8c8d')):
    plt.plot(centers, [m for m, _ in by_bin(ef[arm])], 'o-', color=color, label=f'ExplainFooler, {arm} choice')
plt.axhline(np.mean(rho['[PAD] vs zero baseline']), ls='--', color='#3b6ea5', label='[PAD] vs zero baseline, no attack')
plt.xlabel('Share of replaced words')
plt.ylabel('Spearman with original IG (clipped at 0)')
plt.title('DistilBERT, SST-2, 30 sentences')
plt.ylim(0, 1)
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

As in table 1 of ExplainFooler, choosing the synonym by L2 lowers the correlation with the original attribution more than a random synonym at the same share of replacements, and the class does not change at any step. The number of sentences in the right intervals is below 30: for short sentences with few replaceable words the search ends before the share of replacements reaches 0.5.

## 4. TEF: the zero embedding, the Pearson correlation and the random attack

TEF differs from ExplainFooler in two places. Word order: the embedding of each word in turn is replaced by zero and we look at how much the attribution changes by the same distance $d$ as in the objective below, as in Step 1 of the paper; words are sorted by decreasing change. Objective: $d=1-\frac{\mathrm{PCC}+1}{2}$, where PCC is the Pearson correlation of the attributions of the original and the attacked text. $N=15$ synonyms, the share of replacements up to $\rho_{\max}=0.4$, the IG baseline is zero. The point of comparison RA: a random word order and a random synonym with the same filters.

The summary measure ACC is the area under the curve of mean PCC over the share of replacements $\rho$ from 0 to 0.4. For the curve to be defined at every point, after the search ends the PCC of a sentence stays at the last value reached.

**Task 2. Run TEF and RA on the 30 sentences of `sample`, build the curves of mean PCC on the grid $\rho = 0, 0.05, \dots, 0.4$ and compute the ACC of both attacks by the trapezoidal rule.**

In [ ]:
def tef_order(words):
    label = int(predict([' '.join(words)])[0].argmax())
    a0 = word_attr(words, label, 20, 'zero')
    change = [pcc_objective(a0, word_attr(words, label, 20, 'zero', zero_word=i)) for i in range(len(words))]   # the same distance d as the objective
    return list(np.argsort(change)[::-1])


def pcc_objective(a0, a1):
    return 1 - (pearsonr(a0, a1).statistic + 1) / 2


GRID = np.round(np.arange(0, 0.401, 0.05), 2)


def curve(steps):
    r = np.array([x for x, _, _ in steps])
    pcc = np.array([pearsonr(steps[0][2], a).statistic for _, _, a in steps])
    return np.array([pcc[r <= g + 1e-9][-1] for g in GRID])

In [ ]:
# Your code here

TEF's ACC is lower than RA's: at the same share of replacements the attribution of the TEF text is farther from the original. In the paper, for BERT with IG on Yelp, TEF's ACC came out negative (−0.14 against 0.10 for RA); we have fewer synonym candidates and shorter sentences, so the gap between TEF and RA is smaller.

**Transfer to another explainer.** TEF's replacements were searched against IG. Does the effect hold for Saliency, the absolute gradient with respect to the embedding summed over coordinates? We compute the PCC of the Saliency attributions of the original and the attacked text.

In [ ]:
sal = Saliency(fwd)


def word_saliency(words, target):
    enc, emb, _ = encode(words, 'zero')
    g = sal.attribute(emb, additional_forward_args=(enc['attention_mask'], target), abs=True)
    return to_words(g.sum(-1)[0].detach().cpu().numpy(), enc.word_ids(), len(words))


tr = []
for s0, s1 in tef_texts:
    if s0 == s1:
        continue
    lab = int(predict([s0])[0].argmax())
    tr.append(pearsonr(word_saliency(s0.split(), lab), word_saliency(s1.split(), lab)).statistic)
print(f'sentences where TEF replaced something: {len(tr)}; mean Saliency PCC before and after: {np.mean(tr):.3f}')

In Ivankay et al. the transfer of replacements to another “model and explainer” pair was also weaker: the attack without transfer lowers the PCC more.

## 5. Amnesic probing: remove a property and look at behavior

A probe shows that a property can be extracted from the representation, not that the model uses it. Elazar et al. check use with an intervention: the property is removed from the representation with INLP (Iterative Nullspace Projection), and we look at whether the model's behavior changed.

INLP with our own code: an `sklearn` logistic regression predicts the property from the token-averaged hidden state of DistilBERT layer 5; the representations are projected onto the nullspace of its weights; this repeats until the accuracy of the new probe on the validation split is within one point of the majority-class accuracy, but at most 40 iterations. The projection $P$ is then applied to all positions at the output of layer 5 during the forward pass of the classifier. The Rand control: the same number of random directions.

There are two properties:

- **sentiment**, which the classifier is supposed to predict;
- **length**, whether the sentence is longer than the median of the validation split (in words): a property that is easy to extract but not needed for sentiment.

The training representations are 4000 sentences from the SST-2 train split with at least 6 words (the SST-2 train split also contains short phrases).

**Task 3. For each property print the probe accuracy over INLP iterations, the number of removed directions, the accuracy of the sentiment classifier after removing the property and after removing the same number of random directions.**

In [ ]:
from sklearn.linear_model import LogisticRegression

LAYER = 5
P = None                                  # the projection; None means no intervention


def project(module, inputs, output):
    if P is None:
        return output
    return output @ P                       # in transformers 5 a DistilBERT block returns a tensor


hook = clf.distilbert.transformer.layer[LAYER - 1].register_forward_hook(project)


@torch.no_grad()
def reps_and_preds(sentences, bs=64):
    reps, preds = [], []
    for i in range(0, len(sentences), bs):
        enc = tok(sentences[i:i + bs], return_tensors='pt', padding=True).to(device)
        out = clf(**enc, output_hidden_states=True)
        m = enc['attention_mask'][..., None]
        reps.append(((out.hidden_states[LAYER] * m).sum(1) / m.sum(1)).cpu())
        preds.append(out.logits.argmax(-1).cpu())
    return torch.cat(reps).numpy(), torch.cat(preds).numpy()


train = load_dataset('stanfordnlp/sst2', split='train').shuffle(seed=SEED)
train = train.filter(lambda r: len(r['sentence'].split()) >= 6).select(range(4000))
X_tr, _ = reps_and_preds(train['sentence'])
X_va, pred_va = reps_and_preds(sst['sentence'])
y_va = np.array(sst['label'])
med = np.median([len(s.split()) for s in sst['sentence']])
props = {
    'sentiment': (np.array(train['label']), y_va),
    'length': (np.array([len(s.split()) > med for s in train['sentence']], int),
               np.array([len(s.split()) > med for s in sst['sentence']], int)),
}
print(f'accuracy without intervention: {100 * (pred_va == y_va).mean():.2f}%; median length: {med:.0f} words')

In [ ]:
# Your code here

Two properties give two different pictures, as in Elazar et al. The probe extracts sentence length almost without error, but after its removal the sentiment classifier's accuracy stays at the level of the random control: the property is in the representation, and the model does not use it for its task. INLP does not bring sentiment down to the majority-class level in 40 iterations (at layer 5 it is written in many directions), but removing just these 40 directions lowers the classifier's accuracy to 69.15%, while removing 40 random ones lowers it only to 90.71%: the model uses this property.

Without the Rand control the second conclusion would be wrong: the accuracy drop after any projection could be explained by the loss of 40 dimensions alone.

## 6. An attack on the SAE representation by replacing one token

Li et al. change one token of a prompt so that the SAE representation moves away from the original one, and show that the concept in the activations of the LLM itself is preserved. Here is the same scheme on GPT-2 small and the SAE `gpt2-small-res-jb` on `blocks.7.hook_resid_pre`, the same pair of model and SAE as in the notebook `PRACTICE_sae_lens`. The procedure follows the authors' code (`src/attacks/replacement.py` in AI4LIFE-GROUP/sae_robustness) with their default values:

- **Input**: the prompt with the instruction `"\nThe previous text is about"` appended; the representation is taken at the last token. The authors' code uses exactly this instruction, the paper writes “The previous sentence is about”.
- **Representation and similarity**: the set of the $k$ most active SAE latents and the concept overlap ratio, the share of common indices of two sets. $k$ is the number of nonzero latents of the original prompt, as in the paper.
- **Untargeted population attack**: for every position $t$ of the prompt (except the first token) a separate attack that changes only token $t$. It is 20 GCG steps: the gradient of $-\cos(z,z_0)$ with respect to the one-hot vector of tokens gives 300 candidate tokens for position $t$, 100 of them are drawn at random and checked exactly, and the best one by overlap becomes the new prompt. The result is the best overlap averaged over positions.
- **Point of comparison**: the same 20 steps with 100 random tokens per position, as the authors' `--random` mode.
- **Targeted individual attack**: 5 latents of the unrelated prompt “The stock market fell sharply after the announcement” that are active on it and are not in the set of our prompt. For every position and every latent, up to 10 GCG steps on $\log\mathrm{softmax}(z)$ of this latent; a latent is switched on if its activation becomes greater than zero. The result is the share of switched-on latents averaged over positions.
- **What is preserved**: the authors check it with a linear probe on the LLM activations; here it is simpler: the cosine of the residual stream at the last token before and after the best untargeted replacement.

In [ ]:
from sae_lens import SAE, HookedSAETransformer

hook.remove()                             # the INLP projection is no longer needed
lm = HookedSAETransformer.from_pretrained('gpt2-small', device=device)
sae = SAE.from_pretrained(release='gpt2-small-res-jb', sae_id='blocks.7.hook_resid_pre', device=device)
HOOK = sae.cfg.metadata.hook_name
W_E = lm.W_E
EOS = lm.tokenizer.eos_token_id
SUFFIX = '\nThe previous text is about'   # the instruction of the authors' code
M_TOP, BATCH, ITERS_POP, ITERS_IND, N_LAT = 300, 100, 20, 10, 5   # the defaults of the authors' src/config.py
gen = torch.Generator().manual_seed(SEED)


def last_state(toks):
    with torch.no_grad():
        _, cache = lm.run_with_cache(toks, names_filter=HOOK)
        h = cache[HOOK][:, -1]
        return h, sae.encode(h)


def overlap(s_batch, s_ref):
    ref = set(s_ref.tolist())
    return np.array([len(set(row.tolist()) & ref) / len(ref) for row in s_batch])


def token_grad(toks, objective):
    # gradient of the objective with respect to the one-hot token matrix; the end-of-text token is never proposed, as in the authors' code
    onehot = torch.zeros(toks.shape[1], W_E.shape[0], device=device)
    onehot[torch.arange(toks.shape[1]), toks[0]] = 1
    onehot.requires_grad_(True)
    store = {}
    lm.run_with_hooks(toks, fwd_hooks=[('hook_embed', lambda x, hook: (onehot @ W_E)[None]),
                                       (HOOK, lambda x, hook: store.update(h=x))])
    objective(sae.encode(store['h'][0, -1])).backward()
    g = onehot.grad
    g[:, EOS] = -float('inf')
    return g


def candidates(x, t, objective, random):
    if random:
        return torch.randint(0, W_E.shape[0], (BATCH,), generator=gen).to(device)
    top_m = torch.topk(token_grad(x, objective)[t], M_TOP).indices
    return top_m[torch.randint(0, M_TOP, (BATCH,), generator=gen).to(device)]


def population_attack(toks, n_x1, z0, s0, random=False):
    """The authors' untargeted attack: for every position t of the prompt except the first, ITERS_POP greedy steps that change only token t. Returns the best overlap per position and the best prompts."""
    k, best, best_x = len(s0), [], []
    for t in range(1, n_x1):
        x, b, bx = toks.clone(), 1.0, toks
        for _ in range(ITERS_POP):
            batch = x.repeat(BATCH, 1)
            batch[:, t] = candidates(x, t, lambda z: -torch.nn.functional.cosine_similarity(z, z0, dim=0), random)
            _, zb = last_state(batch)
            ov = overlap(torch.topk(zb, k).indices, s0)
            j = int(ov.argmin())
            x = batch[j:j + 1]
            if ov[j] < b:
                b, bx = ov[j], x
        best.append(b)
        best_x.append(bx)
    return np.array(best), best_x


def individual_attack(toks, n_x1, latents):
    """The authors' targeted individual attack: for every position and every latent up to ITERS_IND GCG steps; a latent is on if its activation is > 0. Returns the share of switched-on latents averaged over positions."""
    rates = []
    for t in range(1, n_x1):
        on = 0
        for f in latents:
            x = toks.clone()
            for _ in range(ITERS_IND):
                batch = x.repeat(BATCH, 1)
                batch[:, t] = candidates(x, t, lambda z: torch.log_softmax(z, 0)[f], False)
                _, zb = last_state(batch)
                rank = (zb > zb[:, f:f + 1]).sum(1)          # rank of the latent: how many latents are more active
                j = int(rank.argmin())
                x = batch[j:j + 1]
                if zb[j, f] > 0:
                    on += 1
                    break
        rates.append(on / len(latents))
    return np.mean(rates)


In [ ]:
PROMPTS = ['How do I bake a chocolate cake at home?', 'The capital of France is a city called',
           'My favourite football team won the match yesterday', 'Write a short poem about the ocean at night',
           'The doctor said that the patient needs more rest', 'Please explain how a computer stores numbers']
UNRELATED = 'The stock market fell sharply after the announcement'
_, z_u = last_state(lm.to_tokens(UNRELATED + SUFFIX))
s_u = torch.topk(z_u[0], int((z_u[0] > 0).sum())).indices          # latents of the unrelated prompt that are active on it
rows_sae = []
for p in PROMPTS:
    n_x1 = lm.to_tokens(p).shape[1]
    toks = lm.to_tokens(p + SUFFIX)
    h0, z0 = last_state(toks)
    s0 = torch.topk(z0[0], int((z0[0] > 0).sum())).indices          # k is the number of nonzero latents of the original prompt
    ov_g, x_g = population_attack(toks, n_x1, z0[0], s0)
    ov_r, _ = population_attack(toks, n_x1, z0[0], s0, random=True)
    j = int(ov_g.argmin())
    h_g, _ = last_state(x_g[j])
    cos_g = torch.nn.functional.cosine_similarity(h_g, h0).item()
    cand = s_u[~torch.isin(s_u, s0)]
    latents = cand[torch.topk(z_u[0, cand], min(N_LAT, len(cand))).indices].tolist()   # the 5 latents most active on the unrelated prompt outside our set
    rate = individual_attack(toks, n_x1, latents)
    rows_sae.append(dict(prompt=p, k=len(s0), gcg=ov_g.mean(), rnd=ov_r.mean(), gcg_best=ov_g.min(), gcg_cos=cos_g, ind=rate))
    print(f'{p} | k = {len(s0)}')
    print(f'   untargeted GCG: overlap averaged over positions {ov_g.mean():.3f}, best {ov_g.min():.3f}, residual-stream cosine {cos_g:.3f} | random tokens: {ov_r.mean():.3f}')
    print(f'   best attacked prompt: {lm.to_string(x_g[j][0, 1:n_x1])!r} | targeted individual: {rate:.2f} of the latents switched on')


**Task 4. Using `rows_sae`, compute the means over the six prompts: the overlap after the untargeted GCG replacement (averaged over positions), the same overlap for random tokens, the residual-stream cosine after the best GCG replacement and the share of latents switched on by the targeted individual attack.**

In [ ]:
# Your code here

Compare the means of task 4. How many of the $k\approx52$ active latents of the original prompt does one token replacement keep, and how much does the residual stream at the last token change? How much further than random search with the same number of checks does GCG move the set? If the gap between them is small, for this pair of model and SAE the set of active latents is fragile by itself.

Compare the share of latents switched on by the targeted individual attack with Li et al.: on Gemma2 and Llama3 their share is above 0.6. Is it higher or lower on GPT-2 small with the SAE `gpt2-small-res-jb`, and does the residual stream change much meanwhile?

## Takeaways

- **The attribution can be shifted without changing the class.** Synonym replacements chosen by the distance between attributions lower the correlation with the original IG attribution more than random synonyms, with the prediction preserved.
- **First measure how much the explanation changes without an adversary.** Changing the IG baseline changes the word ranks comparably with an attack; the number of IG steps barely changes them.
- **Candidates determine the strength of the attack.** With WordNet synonyms there are fewer candidates than with the counter-fitted embeddings of the papers, and the gap between the attack and the random point of comparison is smaller than in the papers.
- **A probe finding a property does not mean the model uses it.** Length is extracted almost without error, but removing it does not change the classifier's behavior; removing sentiment does. Only the control with random directions makes it possible to tell these apart.
- **The SAE representation is more fragile than the activations it is built from.** One token replacement changes the set of active SAE latents more than random search does, while the residual stream changes moderately; the targeted attack with one token switches on part of the chosen latents of an unrelated prompt. The numbers are in the output of task 4.